# 05 · BYOK & 自定义 Provider

目标：在不依赖 GitHub Copilot 订阅的情况下，把 SDK 接到 **Azure OpenAI / OpenAI / Ollama** 等 OpenAI-兼容端点。

> 重要：BYOK **仅支持 key/bearer** 鉴权 —— Entra ID、Managed Identity、第三方 IdP 都不支持。
> 用 BYOK 时 `model` 参数**必填**（SDK 不会再帮你猜默认值）。

## 0. 认证与上游路由全景

SDK 在 **跑 LLM** 这一步决定走 GitHub 还是你的 provider，规则可以画成这样：

```mermaid
flowchart TD
    Start([create_session 调用]) --> HasProvider{传了<br/>provider=...?}
    HasProvider -- 是 --> BYOK["BYOK 路径<br/>CLI 直接打你给的 base_url<br/>(不需要 GitHub 凭证)"]
    HasProvider -- 否 --> GH["GitHub Copilot 路径<br/>需要 GitHub 凭证"]

    GH --> A1{github_token=...?}
    A1 -- 是 --> Use1["用此 token"]
    A1 -- 否 --> A2{COPILOT_GITHUB_TOKEN env?}
    A2 -- 是 --> Use2["用环境变量"]
    A2 -- 否 --> A3{GH_TOKEN / GITHUB_TOKEN?}
    A3 -- 是 --> Use3["用环境变量"]
    A3 -- 否 --> A4{copilot CLI 已 login?}
    A4 -- 是 --> Use4["用本地 OAuth credentials"]
    A4 -- 否 --> Err["❌ 401 / 鉴权失败"]

    BYOK --> Type{provider.type}
    Type -- azure --> AzureURL["base_url = https://&lt;resource&gt;.openai.azure.com<br/>(❗不要带 /openai/v1)<br/>+ azure.api_version"]
    Type -- openai --> OpenAIURL["base_url = https://api.openai.com/v1<br/>(或任何 OpenAI-兼容端点)"]
    Type -- anthropic --> AnthURL["base_url = https://api.anthropic.com"]

    AzureURL --> Wire{wire_api}
    OpenAIURL --> Wire
    Wire -- "completions (默认)" --> ChatAPI["走 Chat Completions API"]
    Wire -- "responses" --> RespAPI["走 Responses API<br/>(Azure 已 GA)"]
```

**BYOK 必知 4 点**：

| 项 | 说明 |
|---|---|
| 鉴权方式 | **仅** 支持 `api_key` / `bearer_token`。**不支持** Entra ID / Managed Identity / 第三方 IdP。生产里用 Key Vault 拉 key 注入。 |
| `model` 必填 | BYOK 下 SDK 不会猜模型名，必须在 `create_session(model=...)` 显式给。 |
| Azure 的 `base_url` 陷阱 | 只写 `https://<res>.openai.azure.com`，**别**带 `/openai/v1` 或部署名。`azure.api_version` 通过单独字段传。 |
| `wire_api` 切换 | `'completions'`（默认 Chat Completions）vs `'responses'`（新协议）。同一端点切，业务代码 0 改动。 |


## 1. ProviderConfig 字段

| 字段 | 说明 |
|---|---|
| `type` | `'openai'` / `'azure'` / `'anthropic'` |
| `base_url` | 端点 host；**Azure 只写 `https://<resource>.openai.azure.com`**，**不要**带 `/openai/v1` |
| `api_key` | API key（Ollama 可省） |
| `bearer_token` | 取代 `api_key`，优先级更高 |
| `wire_api` | `'completions'`（默认） / `'responses'` —— OpenAI/Azure 专用 |
| `azure` | `{'api_version': '2024-10-21'}` 等 Azure 专属选项 |

## 2. Azure OpenAI（cowork_worker 当前主路径）

In [ ]:
import os, asyncio
from dotenv import load_dotenv
from copilot import CopilotClient
from copilot.session import PermissionHandler
from copilot.generated.session_events import AssistantMessageData, SessionIdleData

# 加载 .env 获取 GPT 5.4 端点
load_dotenv()

azure_provider = {
    'type': 'azure',  # 必须是 azure，不是 openai
    'base_url': os.environ['AZURE_OPENAI_ENDPOINT_GPT_5_4'],  # https://<resource>.openai.azure.com
    'api_key': os.environ['AZURE_OPENAI_API_KEY_GPT_5_4'],
    'azure': {'api_version': os.getenv('AZURE_OPENAI_API_VERSION_GPT_5_4', '2025-03-01-preview')},
}

async def run():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
        ) as session:
            done = asyncio.Event()
            def on_event(e):
                if isinstance(e.data, AssistantMessageData):
                    print(e.data.content)
                elif isinstance(e.data, SessionIdleData):
                    done.set()
            session.on(on_event)
            await session.send('用一句话自我介绍。')
            await done.wait()

await run()

### 2.1 实战：切换 `wire_api` 观察请求形态

`wire_api` 决定 SDK 跟 Azure / OpenAI 端点之间用 **Chat Completions** 协议还是 **Responses API** 协议。两者一对一切换，无需改业务代码 —— 适合做兼容性 / 性能 A/B。

下面跑两次同样的 prompt，分别强制走 `'completions'`（默认）和 `'responses'`，对比延迟。

In [ ]:
# 💡 演示：同一个 Azure 端点，两种 wire_api 切换，记录耗时
import time

PROMPT = '用 8 个字总结你能做什么。'

async def _send_and_time(provider_cfg, label):
    t0 = time.time()
    text = {'value': None}
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=provider_cfg,
        ) as session:
            resp = await session.send_and_wait(PROMPT, timeout=60.0)
            if resp and hasattr(resp.data, 'content'):
                text['value'] = resp.data.content
    dt = time.time() - t0
    print(f'[⚡ {label:<24}] {dt:5.2f}s  → {text["value"]!r}')
    return dt

# A: wire_api='completions'（默认 Chat Completions 协议）
prov_completions = {**azure_provider, 'wire_api': 'completions'}
# B: wire_api='responses'（新的 Responses API 协议，Azure 已 GA）
prov_responses = {**azure_provider, 'wire_api': 'responses'}

t_c = await _send_and_time(prov_completions, 'wire_api=completions')
t_r = await _send_and_time(prov_responses,   'wire_api=responses')

print(f'\n📊 耗时对比: completions={t_c:.2f}s  responses={t_r:.2f}s '
      f'(差 {abs(t_c - t_r):.2f}s)')

## 3. OpenAI / 任何 OpenAI 兼容端点

In [ ]:
openai_provider = {
    'type': 'openai',
    'base_url': 'https://api.openai.com/v1',
    'api_key': os.environ['OPENAI_API_KEY'],
}
# 然后 model='gpt-4o' 等

## 4. Ollama（本地无 key）

In [ ]:
ollama_provider = {
    'type': 'openai',
    'base_url': 'http://localhost:11434/v1',
    # api_key 不需要
}
# model='deepseek-coder-v2:16b' 等

## 5. 切换模型的 3 种正确姿势

cowork_worker 用 `PER_TURN_MODEL` contextvar 全局可变 —— 在 SDK 里有 3 种**官方**做法：

| 姿势 | 何时用 | 代码 |
|---|---|---|
| **A. 每 session 一个 model**（并发友好） | 一个 worker 同时服务多个 workspace | 每次 `create_session(model=...)` 时绑定，独立 history |
| **B. `session.set_model(...)` 中途换 model** ⭐ | 同一对话内切换"小模型起步 → 大模型收尾"等场景 | 历史保留，下一条 send 起生效 |
| **C. session 池子** | 高频请求 + 模型组合稳定 | 用 `(workspace_id, model)` 作 key 缓存 session |

```python
sessions = {}
async def get_session(client, workspace_id, model):
    key = (workspace_id, model)
    if key not in sessions:
        sessions[key] = await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=model, provider=azure_provider,
        )
    return sessions[key]
```

### 5.0 实战：`session.set_model()` 中途换模型 + 保留 history

> SDK 0.3.0 起 `CopilotSession.set_model(model, reasoning_effort=...)` 真实可用——**历史会话保留**，下一条 `send` 起新模型生效。下方 cell 在同一个 session 内先让 mini 接收"我叫张三"，再切到 5.4 让它"回忆我叫什么"。

In [ ]:
# 💡 演示：单 session 内 set_model() 切换 model，验证 history 是否保留
# 注意：BYOK 模式下 set_model 的 model_id 必须是 *当前 provider 端* 认得的部署名
# （我们这里前后都用同一个 Azure provider，只能切换该 provider 上有效的 model id）

MINI_MODEL = os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4')   # 起手
BIG_MODEL  = os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4')   # 收尾


async def run_set_model_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MINI_MODEL,
            provider=azure_provider,
        ) as session:
            # —— 第 1 回合：起手 model，先告诉它我的名字 ——
            r1 = await session.send_and_wait(
                '记住一个事实：我叫张三，今年 28 岁。请用一句话确认你已经记住。',
                timeout=60.0,
            )
            print(f'[第 1 回合 model={MINI_MODEL}]\n   → {r1.data.content if r1 else None}\n')

            # —— 中途换 model（不重建 session，history 保留）——
            print(f'⚙️  session.set_model({BIG_MODEL!r})')
            await session.set_model(BIG_MODEL)

            # —— 第 2 回合：在新 model 下让它回忆名字 ——
            r2 = await session.send_and_wait(
                '我刚才告诉你我叫什么名字、多少岁？请如实回答。',
                timeout=60.0,
            )
            print(f'\n[第 2 回合 model={BIG_MODEL} after set_model]\n   → {r2.data.content if r2 else None}')

            # —— 内省：从 session 直接读 history ——
            msgs = await session.get_messages()
            print(f'\n[📚 session.get_messages() 共 {len(msgs)} 条消息]')


await run_set_model_demo()

### 5.1 实战：两个 session 并发跑两个模型，对比延迟和回复

这正是 cowork_worker 里 `PER_TURN_MODEL` 想做的事，但在 SDK 里干净得多：
- 每个 `create_session(model=...)` 就是一个独立 session 实例
- 用 `asyncio.gather` 并发跑，两个模型**同时**出回复
- 没有 contextvar、没有 race condition

In [ ]:
# 💡 演示：用 asyncio.gather 并发跑两个 model，对比延迟 + 回复差异
# 都用 Azure；endpoint 也分别按模型从 .env 取
import time


def _provider_for(suffix: str) -> dict:
    """根据 env 后缀（GPT_5 / GPT_5_4 / GPT_4o ...）构造 Azure provider 配置。"""
    return {
        'type': 'azure',
        'base_url': os.environ[f'AZURE_OPENAI_ENDPOINT_{suffix}'],
        'api_key': os.environ[f'AZURE_OPENAI_API_KEY_{suffix}'],
        'azure': {
            'api_version': os.getenv(
                f'AZURE_OPENAI_API_VERSION_{suffix}',
                '2025-03-01-preview',
            ),
        },
    }


async def ask(model_env_suffix: str, prompt: str) -> tuple[str, float, str | None]:
    """对某个 model + endpoint 发一条 prompt，返回 (model_label, 延迟s, 文本)。"""
    model = os.environ[f'AZURE_OPENAI_MODEl_{model_env_suffix}']
    t0 = time.time()
    text = None
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=model,
            provider=_provider_for(model_env_suffix),
        ) as session:
            resp = await session.send_and_wait(prompt, timeout=60.0)
            if resp and hasattr(resp.data, 'content'):
                text = resp.data.content
    return model, time.time() - t0, text


PROMPT = '请用 12 个字内回答：什么是 BYOK？'

# ✨ 并发！而非串行
results = await asyncio.gather(
    ask('GPT_5_4', PROMPT),   # 主力
    ask('GPT_5',   PROMPT),   # 老一代
)

print('=' * 60)
for model, dt, text in results:
    print(f'\n[🤖 {model}]  ({dt:.2f}s)')
    print(f'   {text}')
print('=' * 60)
print(f'\n📊 并发总耗时 ≈ max({results[0][1]:.2f}, {results[1][1]:.2f}) = '
      f'{max(r[1] for r in results):.2f}s '
      f'(串行至少要 {sum(r[1] for r in results):.2f}s)')

In [ ]:
# 💡 演示：list_models() 列出当前 client 可见的模型清单 + 关键能力字段
# 注意：BYOK 模式下，list_models 走的是 *默认 GitHub Copilot 鉴权*（不是你的 Azure provider），
# 所以它列出来的是 GitHub Copilot 提供的模型清单，而非你 Azure 资源里的部署。
#
# ⚠️ SDK 0.3.0 有个解析 bug：某些模型 billing 缺 multiplier 字段，会让 client.list_models() 整批崩。
#    本 cell 直接走底层 JSON-RPC 绕过这个序列化 bug。
async with CopilotClient() as client:
    raw = await client._rpc._client.request('models.list', {})
    models = raw.get('models', [])
    print(f'共 {len(models)} 个模型可用：\n')
    print(f'{"id":<32} {"名称":<32} {"reasoning?":<10} {"max_input"}')
    print('-' * 100)
    for m in models:
        cap = m.get('capabilities', {})
        max_in = (
            cap.get('maxInputTokens')
            or cap.get('contextWindow')
            or cap.get('supports', {}).get('maxInputTokens')
            or '—'
        )
        reasoning = 'yes' if m.get('supportedReasoningEfforts') else 'no'
        print(f'{m.get("id",""):<32} {m.get("name","")[:30]:<32} {reasoning:<10} {max_in}')

### 5.5 实战：session 持久化 + `resume_session` 跨进程续聊

默认情况（`infinite_sessions.enabled=True`，开箱即开）下，每个 session 都被完整写到 `~/.copilot/session-state/<uuid>/`：

```
~/.copilot/
├── session-store.db                  ← SQLite 索引
└── session-state/<uuid>/
    ├── events.jsonl                  ← 完整事件流（append-only）
    ├── workspace.yaml                ← 元数据
    ├── checkpoints/  files/  research/
```

本节做 4 件事，全部跑给你看：
1. 第 1 次：创建 session、记下 `session.session_id`、塞一条事实进去
2. **完全断开**（退出 `async with`，相当于进程退出）
3. 第 2 次：`client.resume_session(my_id, ...)` 复活同一个 session，问它"我叫啥"——history 全在
4. 直接读 `~/.copilot/session-state/<my_id>/workspace.yaml` + `sqlite3` 查 `session-store.db`，证明数据确实在磁盘上

> ⚠️ resume 时 `provider` / `on_permission_request` 等**必须重新传**（SDK 不会从磁盘恢复 BYOK key），但 history 由 CLI 从 `events.jsonl` 自动回放，无需你管。

完整说明见 [`docs/session-persistence.md`](docs/session-persistence.md)。


In [ ]:
# 💡 演示：第 1 次会话 —— 创建、记下 session_id、塞一条事实
MODEL = os.environ['AZURE_OPENAI_MODEl_GPT_5_4']

async def step1_create_and_remember() -> str:
    """创建 session，告诉它一个独特事实，返回 session_id 供下一步复用。"""
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=MODEL,
            provider=azure_provider,
        ) as session:
            sid = session.session_id
            print(f'📌 session_id = {sid}')
            resp = await session.send_and_wait(
                '请记住一个事实：我叫李雷，今年 35 岁，最喜欢的颜色是紫色。一句话确认。',
                timeout=60.0,
            )
            text = resp.data.content if (resp and hasattr(resp.data, 'content')) else None
            print(f'🤖 {text}')
            return sid

# 此处一旦退出 async with，client 子进程销毁、session 在内存的引用全部释放——
# 完全等价于"进程退出"。只有磁盘上 ~/.copilot/session-state/<sid>/ 的事件流留下来。
my_session_id = await step1_create_and_remember()
print(f'\n✅ 第 1 步完成；session 已在内存里销毁，仅磁盘留痕。\n   id = {my_session_id}')


In [ ]:
# 💡 演示：磁盘验证 —— 这个 session 的事件确实落盘了
from pathlib import Path
import sqlite3, json, os as _os

state_dir = Path.home() / '.copilot' / 'session-state' / my_session_id
print(f'📁 session 目录：{state_dir}')
print(f'   存在？ {state_dir.exists()}')
print()

if state_dir.exists():
    print('📂 目录内容：')
    for p in sorted(state_dir.iterdir()):
        size = p.stat().st_size if p.is_file() else sum(
            f.stat().st_size for f in p.rglob('*') if f.is_file()
        )
        kind = 'DIR ' if p.is_dir() else 'FILE'
        print(f'   [{kind}] {p.name:<30} {size:>8} B')

    # workspace.yaml
    ws = state_dir / 'workspace.yaml'
    if ws.exists():
        print(f'\n📄 workspace.yaml:\n{ws.read_text()}')

    # events.jsonl 头一条事件
    ev = state_dir / 'events.jsonl'
    if ev.exists():
        first = ev.read_text().splitlines()[0]
        first_obj = json.loads(first)
        print(f'📜 events.jsonl 第 1 条事件 type = {first_obj.get("type")}')
        print(f'   总事件数 = {sum(1 for _ in ev.read_text().splitlines())}')

# SQLite 索引侧
db = Path.home() / '.copilot' / 'session-store.db'
if db.exists():
    con = sqlite3.connect(db)
    row = con.execute(
        'SELECT id, substr(cwd,-50), branch, substr(summary,1,40), updated_at '
        'FROM sessions WHERE id = ?',
        (my_session_id,),
    ).fetchone()
    con.close()
    print(f'\n🗄️  session-store.db 索引行：\n   {row}')


In [ ]:
# 💡 演示：第 2 次会话 —— resume_session 复活，验证 history 还在
async def step2_resume_and_recall(sid: str):
    """用同一个 session_id 复活，问它'我叫啥'——history 应该全在。"""
    async with CopilotClient() as client:
        async with await client.resume_session(
            sid,
            on_permission_request=PermissionHandler.approve_all,
            provider=azure_provider,            # ← BYOK key 不会从磁盘恢复，必须重传
            # model 不传就沿用原 session 的 model
            # 想换 model：传 model='claude-opus-4.7' 即可
        ) as session:
            # 直接问之前塞进去的事实
            resp = await session.send_and_wait(
                '我刚才告诉你我叫什么名字、几岁、喜欢什么颜色？请一句话回答。',
                timeout=60.0,
            )
            text = resp.data.content if (resp and hasattr(resp.data, 'content')) else None
            print(f'🤖 (resume 后回答) {text}')

            # get_messages() 返回 list[SessionEvent]（.type / .data / .timestamp）
            msgs = await session.get_messages()
            print(f'\n📜 session.get_messages() 当前 {len(msgs)} 条事件')
            for i, ev in enumerate(msgs[:8]):
                etype = getattr(ev, 'type', '?')
                data = getattr(ev, 'data', None)
                # data 可能是各种 dataclass：UserMessageData / AssistantMessageData / SystemMessageData 等
                content = getattr(data, 'content', None) or getattr(data, 'role', '')
                snippet = str(content)[:60].replace('\n', ' ')
                print(f'   [{i:>2}] {etype:<28} {snippet}')
            if len(msgs) > 8:
                print(f'   ... 还有 {len(msgs) - 8} 条')

await step2_resume_and_recall(my_session_id)
print('\n✅ 第 2 步完成；session 已跨"进程"复活，history 经 events.jsonl 自动回放。')


## 6.1 实战：故意把 Provider 配错，看 SDK 怎么报错

新人最容易踩的三个坑：

| 错法 | 真实报错 | 修法 |
|---|---|---|
| `type='openai'` 配 Azure URL | `HTTP 404 Resource not found` | 改 `type='azure'` |
| `base_url` 带了 `/openai/v1` | `HTTP 404 Resource not found` | 只保留 `https://<resource>.openai.azure.com` |
| 漏了 `azure.api_version` | ⚠️ **不报错**（SDK 用内置默认值，可能跟旧版 API 兼容）| 仍建议显式声明，避免哪天 SDK 默认值变了 |

下面把 3 个错法各跑一次，捕获异常打印——你以后看到类似错误就能秒识别。

In [ ]:
# 💡 演示：3 种错误 provider 配置 + 真实错误捕获
MODEL = os.environ['AZURE_OPENAI_MODEl_GPT_5_4']
GOOD_BASE = os.environ['AZURE_OPENAI_ENDPOINT_GPT_5_4']
KEY = os.environ['AZURE_OPENAI_API_KEY_GPT_5_4']

cases = {
    '❌ type=openai 配 Azure URL': {
        'type': 'openai',     # 错：应为 azure
        'base_url': GOOD_BASE,
        'api_key': KEY,
    },
    '❌ base_url 多了 /openai/v1': {
        'type': 'azure',
        'base_url': GOOD_BASE.rstrip('/') + '/openai/v1',   # 错：应只到 host
        'api_key': KEY,
        'azure': {'api_version': '2025-03-01-preview'},
    },
    '❌ 漏了 azure.api_version': {
        'type': 'azure',
        'base_url': GOOD_BASE,
        'api_key': KEY,
        # 故意漏 azure 字段
    },
}


async def try_provider(label, prov):
    try:
        async with CopilotClient() as client:
            async with await client.create_session(
                on_permission_request=PermissionHandler.approve_all,
                model=MODEL,
                provider=prov,
            ) as session:
                resp = await session.send_and_wait('hi', timeout=15.0)
                txt = resp.data.content if (resp and hasattr(resp.data, 'content')) else None
                print(f'[{label}]  ✅ 居然成功了？→ {txt!r}')
    except Exception as e:
        msg = str(e)
        # 截断超长 stack
        if len(msg) > 200:
            msg = msg[:200] + ' …'
        print(f'[{label}]\n   → {type(e).__name__}: {msg}\n')


for label, prov in cases.items():
    await try_provider(label, prov)

## 7. Takeaways

- Azure 必须 `type='azure'` 且 `base_url` 只到 host —— 详见 §6.1 的 3 种错配 vs 报错对照
- `wire_api` 在 `completions` / `responses` 间一行切换 —— 见 §2.1
- BYOK 不支持 Managed Identity —— 想用 Entra 必须在你的服务里换 key/SAS
- **每个 session 绑一个 model**，并发用 `asyncio.gather` —— 见 §5.1 GPT-5.4 vs GPT-5 并发实战；这是替代 cowork_worker `PER_TURN_MODEL` 的官方答案
- `client.list_models()` 列出 GitHub Copilot 鉴权下可见模型（**不含**你 Azure 部署的 deployment）—— 见 §6